In [ ]:
from datasets import load_dataset

# Using sample-10BT for manageability; swap name for a specific CC dump
# (e.g. 'CC-MAIN-2024-46') or remove name entirely for the full 25.9B-row dataset
fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", streaming=True)

## Exploratory Analysis

In [ ]:
# Peek at the first record to see available fields
sample = next(iter(fw["train"]))
print("Fields:", list(sample.keys()))
print()
for k, v in sample.items():
    preview = str(v)[:200] + "..." if len(str(v)) > 200 else str(v)
    print(f"  {k}: {preview}")

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt

# Randomly select shards so samples are spread across the full stream.
# sample-10BT has ~500 shards; adjust TOTAL_SHARDS if using a different config.
N_SHARDS_TO_SAMPLE = 50
RECORDS_PER_SHARD  = 100
TOTAL_SHARDS       = 500
SEED               = 42

random.seed(SEED)
shard_indices = sorted(random.sample(range(TOTAL_SHARDS), N_SHARDS_TO_SAMPLE))

timestamps = []
for shard_idx in shard_indices:
    shard_ds = fw["train"].shard(num_shards=TOTAL_SHARDS, index=shard_idx)
    count = 0
    for record in shard_ds:
        if count >= RECORDS_PER_SHARD:
            break
        ts = record.get("date")
        if ts:
            timestamps.append(ts)
            count += 1
    print(f"  Shard {shard_idx:4d}: {count} records  (running total: {len(timestamps)})")

print(f"\nSampled {len(timestamps)} records across {N_SHARDS_TO_SAMPLE} shards")

# Parse timestamps (ISO format)
dates = pd.to_datetime(timestamps, utc=True)
s = pd.Series(dates)
by_year = s.dt.year.value_counts().sort_index()
by_ym   = s.dt.to_period("M").value_counts().sort_index()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

by_year.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title(f"Records by Year  (n={len(timestamps):,}, {N_SHARDS_TO_SAMPLE} random shards)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

by_ym_ts = by_ym.copy()
by_ym_ts.index = by_ym_ts.index.to_timestamp()
by_ym_ts.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="none", width=1.0)
axes[1].set_title(f"Records by Year-Month  (n={len(timestamps):,}, {N_SHARDS_TO_SAMPLE} random shards)")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Count")
n_ticks = len(by_ym_ts)
step = max(1, n_ticks // 12)
axes[1].set_xticks(range(0, n_ticks, step))
axes[1].set_xticklabels(
    [f"{by_ym_ts.index[i].year}-{str(by_ym_ts.index[i].month).zfill(2)}"
     for i in range(0, n_ticks, step)],
    rotation=45, ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# Date range and distribution stats
print("=== Date Range ===")
print(f"  Earliest : {s.min()}")
print(f"  Latest   : {s.max()}")
print(f"  Span     : {s.max() - s.min()}")
print()
print("=== Distribution by Year ===")
for year, count in by_year.items():
    bar = "\u2588" * (count * 40 // by_year.max())
    print(f"  {year}  {count:>6}  {bar}")
print()
print("=== Distribution by Month (all years combined) ===")
by_month = s.dt.month.value_counts().sort_index()
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for month, count in by_month.items():
    bar = "\u2588" * (count * 40 // by_month.max())
    print(f"  {month_names[month-1]}  {count:>6}  {bar}")